# FinMark Corporation — Feature Engineering

This notebook builds **model-ready features** from the cleaned outputs produced in:
- `FinMark_Preprocessing_EDA.ipynb`

## Inputs (from the cleaning layer)
- `cleaned/transactions_cleaned.csv`
- `cleaned/feedback_customer_summary.csv`
- `cleaned/products_cleaned.csv` (contextual; not joinable to transactions in the provided data)

## Outputs (saved locally)
- `features/customer_features.csv` (one row per customer; unscaled)
- `features/customer_features_scaled.csv` (numeric features standardized; ready for clustering)
- `features/feature_dictionary.csv` (short description per feature)
- `features/transactions_time_features.csv` (**transaction-level** time-normalized features for documentation/EDA)
- `features/merged_master_transactions.csv` (**model-friendly** transaction-level master table + customer summary; no duplication)

## 0) Setup

### Scope
- Create **customer-level** features appropriate for segmentation/clustering (one row per `Customer_ID`)
- Add an additional **transaction-level time feature export** to demonstrate date/time normalization and temporal transformations


In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 140)
pd.options.display.float_format = "{:,.2f}".format


## 1) Load cleaned datasets

We load data from the `cleaned/` folder to ensure feature engineering is based on the finalized cleaned layer.


In [2]:

DATA_DIR_CANDIDATES = [Path("cleaned"), Path("/mnt/data/cleaned"), Path("."), Path("/mnt/data")]

def resolve_path(filename: str) -> Path:
    for d in DATA_DIR_CANDIDATES:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename} in {DATA_DIR_CANDIDATES}")

txn_path    = resolve_path("transactions_cleaned.csv")
fb_agg_path = resolve_path("feedback_customer_summary.csv")
prod_path   = resolve_path("products_cleaned.csv")

transactions = pd.read_csv(txn_path)
feedback_agg = pd.read_csv(fb_agg_path)
products     = pd.read_csv(prod_path)

# Ensure datetime is parsed (CSV may store it as string)
transactions["Transaction_Date"] = pd.to_datetime(transactions["Transaction_Date"], errors="coerce")

print("Loaded cleaned datasets:")
print("- transactions:", transactions.shape, "from", txn_path)
print("- feedback_agg:", feedback_agg.shape, "from", fb_agg_path)
print("- products:", products.shape, "from", prod_path)

display(transactions.head())
display(feedback_agg.head())


Loaded cleaned datasets:
- transactions: (5000, 8) from cleaned/transactions_cleaned.csv
- feedback_agg: (1000, 7) from cleaned/feedback_customer_summary.csv
- products: (10, 5) from cleaned/products_cleaned.csv


,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Amount,Transaction_Type,amount_was_missing,amount_outlier_flag,Transaction_Amount_Raw
0,1,393,2023-01-01 00:00:00,"3,472.00",Purchase,0,0,"3,472.00"
1,2,826,2023-01-01 01:00:00,"2,485.00",Bill Payment,1,0,"2,485.00"
2,3,916,2023-01-01 02:00:00,10.00,Purchase,0,0,10.00
3,4,109,2023-01-01 03:00:00,72.00,Investment,0,0,72.00
4,5,889,2023-01-01 04:00:00,"1,793.00",Investment,0,0,"1,793.00"


,Customer_ID,feedback_count,satisfaction_mean,satisfaction_std,ltr_mean,ltr_std,top_comment
0,1,2,8.50,2.12,9.00,0.00,Very Satisfied
1,2,3,4.33,2.31,5.00,2.00,Very Satisfied
2,3,3,8.33,2.08,3.67,3.79,Very Satisfied
3,4,6,5.50,2.07,3.33,2.50,Needs Improvement
4,5,9,6.44,2.01,5.67,2.87,Unsatisfactory


## 2) Transaction-level time normalization (intermediate)

### What this adds
These columns are created to support customer aggregation and to document date/time normalization:
- week/year extraction (ISO week + calendar year)
- hour extraction
- time-of-day buckets

> These are **intermediate** fields used downstream and also exported later as `transactions_time_features.csv`.


In [3]:

tx = transactions.copy()

# Core date parts
tx["Transaction_Day"] = tx["Transaction_Date"].dt.date
tx["Transaction_Month"] = tx["Transaction_Date"].dt.month
tx["Transaction_Year"] = tx["Transaction_Date"].dt.year

# ISO week/year (more stable week definition across years)
iso = tx["Transaction_Date"].dt.isocalendar()
tx["Transaction_Week"] = iso.week.astype(int)
tx["Transaction_ISOYear"] = iso.year.astype(int)

# Hour extraction (if datetime contains hour; if not, hour defaults to 0)
tx["Transaction_Hour"] = tx["Transaction_Date"].dt.hour.fillna(0).astype(int)

def time_of_day_bucket(h: int) -> str:
    if 5 <= h <= 11:
        return "Morning"
    if 12 <= h <= 16:
        return "Afternoon"
    if 17 <= h <= 20:
        return "Evening"
    return "Night"

tx["Transaction_TimeOfDay"] = tx["Transaction_Hour"].apply(time_of_day_bucket)

display(tx[["Customer_ID","Transaction_Date","Transaction_Day","Transaction_Week","Transaction_Year","Transaction_Hour","Transaction_TimeOfDay"]].head())


,Customer_ID,Transaction_Date,Transaction_Day,Transaction_Week,Transaction_Year,Transaction_Hour,Transaction_TimeOfDay
0,393,2023-01-01 00:00:00,2023-01-01,52,2023,0,Night
1,826,2023-01-01 01:00:00,2023-01-01,52,2023,1,Night
2,916,2023-01-01 02:00:00,2023-01-01,52,2023,2,Night
3,109,2023-01-01 03:00:00,2023-01-01,52,2023,3,Night
4,889,2023-01-01 04:00:00,2023-01-01,52,2023,4,Night


## 3) Daily / weekly spend trend (grouped correctly)

### Definition used in this notebook
- **Daily_Spend_Trend:** total spend per **Customer_ID + Transaction_Day**
- **Weekly_Spend_Trend:** total spend per **Customer_ID + ISO Week (Transaction_ISOYear + Transaction_Week)**

This grouping avoids inflated values and ensures each transaction row receives the correctly aggregated daily and weekly totals.


In [4]:

# Daily total spend per customer-day
daily_totals = (
    tx.groupby(["Customer_ID", "Transaction_Day"], as_index=False)
      .agg(Daily_Spend_Trend=("Transaction_Amount", "sum"))
)

# Weekly total spend per customer-ISOweek (week + isoyear)
weekly_totals = (
    tx.groupby(["Customer_ID", "Transaction_ISOYear", "Transaction_Week"], as_index=False)
      .agg(Weekly_Spend_Trend=("Transaction_Amount", "sum"))
)

# Merge back to transaction rows
tx = tx.merge(daily_totals, on=["Customer_ID", "Transaction_Day"], how="left") \
       .merge(weekly_totals, on=["Customer_ID", "Transaction_ISOYear", "Transaction_Week"], how="left")

display(tx[["Customer_ID","Transaction_Day","Transaction_Week","Transaction_ISOYear","Daily_Spend_Trend","Weekly_Spend_Trend"]].head())


,Customer_ID,Transaction_Day,Transaction_Week,Transaction_ISOYear,Daily_Spend_Trend,Weekly_Spend_Trend
0,393,2023-01-01,52,2022,"3,472.00","3,472.00"
1,826,2023-01-01,52,2022,"2,485.00","2,485.00"
2,916,2023-01-01,52,2022,"1,062.00","1,062.00"
3,109,2023-01-01,52,2022,72.00,72.00
4,889,2023-01-01,52,2022,"1,793.00","1,793.00"


## 4) Export: transaction-level time features (documentation layer)

This export is **not** used directly in the customer-level clustering table.  
Its role is to demonstrate date/time normalization and to support additional EDA or reporting.

Output:
- `features/transactions_time_features.csv`


In [5]:

# Keep a focused set of columns for the export
tx_time_features = tx[[
    "Transaction_ID",
    "Customer_ID",
    "Transaction_Date",
    "Transaction_Type",
    "Transaction_Amount",
    "Transaction_Day",
    "Transaction_Week",
    "Transaction_Month",
    "Transaction_Year",
    "Transaction_Hour",
    "Transaction_TimeOfDay",
    "Daily_Spend_Trend",
    "Weekly_Spend_Trend",
]].copy()

# Round numeric columns for clean presentation
num_cols = tx_time_features.select_dtypes(include=[np.number]).columns
tx_time_features[num_cols] = tx_time_features[num_cols].round(2)

display(tx_time_features.head())


,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Type,Transaction_Amount,Transaction_Day,Transaction_Week,Transaction_Month,Transaction_Year,Transaction_Hour,Transaction_TimeOfDay,Daily_Spend_Trend,Weekly_Spend_Trend
0,1,393,2023-01-01 00:00:00,Purchase,"3,472.00",2023-01-01,52,1,2023,0,Night,"3,472.00","3,472.00"
1,2,826,2023-01-01 01:00:00,Bill Payment,"2,485.00",2023-01-01,52,1,2023,1,Night,"2,485.00","2,485.00"
2,3,916,2023-01-01 02:00:00,Purchase,10.00,2023-01-01,52,1,2023,2,Night,"1,062.00","1,062.00"
3,4,109,2023-01-01 03:00:00,Investment,72.00,2023-01-01,52,1,2023,3,Night,72.00,72.00
4,5,889,2023-01-01 04:00:00,Investment,"1,793.00",2023-01-01,52,1,2023,4,Night,"1,793.00","1,793.00"


## 5) Model-friendly master merged dataset (no duplication)

### Why this dataset exists
For reporting and some downstream analytics, it is often useful to have a **single “master” table** at the *transaction level*.

This version is **model-friendly** because it does **not** multiply rows based on feedback events:
- We merge transactions with **customer-level feedback summary** (`feedback_customer_summary.csv`).
- Since the feedback file has **one row per customer**, the merged dataset remains **one row per transaction**.

### Output
- `features/merged_master_transactions.csv`


In [6]:

# Customer total spend (single source of truth from transactions)
customer_total_spend = (
    tx_time_features.groupby("Customer_ID", as_index=False)["Transaction_Amount"]
                   .sum()
                   .rename(columns={"Transaction_Amount": "Total_Spend"})
)

# Model-friendly merge: transaction rows + customer-level feedback summary
merged_master_transactions = (
    tx_time_features
        .merge(feedback_agg, on="Customer_ID", how="left")
        .merge(customer_total_spend, on="Customer_ID", how="left")
)

# Presentation cleanup (avoid float-looking ints)
for c in ["Transaction_Week", "Transaction_Month", "Transaction_Year", "Transaction_Hour"]:
    if c in merged_master_transactions.columns:
        merged_master_transactions[c] = pd.to_numeric(merged_master_transactions[c], errors="coerce").astype("Int64")

# Round numeric columns for clean presentation
num_cols = merged_master_transactions.select_dtypes(include=[np.number]).columns
merged_master_transactions[num_cols] = merged_master_transactions[num_cols].round(2)

print("merged_master_transactions shape:", merged_master_transactions.shape)
display(merged_master_transactions.head())


merged_master_transactions shape: (5000, 20)


,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Type,Transaction_Amount,Transaction_Day,Transaction_Week,Transaction_Month,Transaction_Year,Transaction_Hour,Transaction_TimeOfDay,Daily_Spend_Trend,Weekly_Spend_Trend,feedback_count,satisfaction_mean,satisfaction_std,ltr_mean,ltr_std,top_comment,Total_Spend
0,1,393,2023-01-01 00:00:00,Purchase,"3,472.00",2023-01-01,52,1,2023,0,Night,"3,472.00","3,472.00",1,1.00,0.00,3.00,0.00,Excellent,"22,365.00"
1,2,826,2023-01-01 01:00:00,Bill Payment,"2,485.00",2023-01-01,52,1,2023,1,Night,"2,485.00","2,485.00",6,6.17,2.93,5.50,2.88,Needs Improvement,"6,931.00"
2,3,916,2023-01-01 02:00:00,Purchase,10.00,2023-01-01,52,1,2023,2,Night,"1,062.00","1,062.00",7,2.71,2.06,5.00,3.21,Needs Improvement,"18,406.00"
3,4,109,2023-01-01 03:00:00,Investment,72.00,2023-01-01,52,1,2023,3,Night,72.00,72.00,3,3.00,1.00,7.00,2.00,Good Service,"7,492.00"
4,5,889,2023-01-01 04:00:00,Investment,"1,793.00",2023-01-01,52,1,2023,4,Night,"1,793.00","1,793.00",5,5.00,3.08,5.60,2.70,Good Service,"5,772.00"


## 6) Customer-level spending features (UNCHANGED)

The customer-level feature table remains consistent with the prior version:  
we aggregate transactions to customer-level and merge feedback summaries.


In [7]:

# Spending aggregates
spend = tx.groupby("Customer_ID").agg(
    txn_count=("Transaction_ID", "size"),
    total_spend=("Transaction_Amount", "sum"),
    avg_spend=("Transaction_Amount", "mean"),
    median_spend=("Transaction_Amount", "median"),
    min_spend=("Transaction_Amount", "min"),
    max_spend=("Transaction_Amount", "max"),
    std_spend=("Transaction_Amount", "std"),
).reset_index()

spend["std_spend"] = spend["std_spend"].fillna(0)

# Coefficient of variation (avoid div by 0)
spend["cv_spend"] = np.where(spend["avg_spend"] > 0, spend["std_spend"] / spend["avg_spend"], 0)

# Cleaning-related rates (if columns exist)
if "amount_was_missing" in tx.columns:
    miss_rate = tx.groupby("Customer_ID")["amount_was_missing"].mean().reset_index(name="pct_amount_imputed")
    spend = spend.merge(miss_rate, on="Customer_ID", how="left")
else:
    spend["pct_amount_imputed"] = 0

if "amount_outlier_flag" in tx.columns:
    out_rate = tx.groupby("Customer_ID")["amount_outlier_flag"].mean().reset_index(name="pct_amount_outlier")
    spend = spend.merge(out_rate, on="Customer_ID", how="left")
else:
    spend["pct_amount_outlier"] = 0

display(spend.head())


,Customer_ID,txn_count,total_spend,avg_spend,median_spend,min_spend,max_spend,std_spend,cv_spend,pct_amount_imputed,pct_amount_outlier
0,1,6,"16,836.00","2,806.00","2,771.00",156.00,"4,993.00","2,062.31",0.73,0.00,0.00
1,2,2,"4,907.00","2,453.50","2,453.50","2,057.00","2,850.00",560.74,0.23,0.00,0.00
2,3,1,"1,538.00","1,538.00","1,538.00","1,538.00","1,538.00",0.00,0.00,0.00,0.00
3,4,2,"8,295.00","4,147.50","4,147.50","3,559.00","4,736.00",832.26,0.20,0.00,0.00
4,5,5,"14,798.00","2,959.60","2,305.00","1,508.00","4,878.00","1,386.58",0.47,0.00,0.00


## 7) Customer-level temporal behavior features (recency, tenure, timing) (UNCHANGED)

In [8]:

# Recency & tenure
overall_last_date = tx["Transaction_Date"].max()

tenure = tx.groupby("Customer_ID").agg(
    first_txn=("Transaction_Date", "min"),
    last_txn=("Transaction_Date", "max"),
    active_days=("Transaction_Day", "nunique"),
).reset_index()

tenure["tenure_days"] = (tenure["last_txn"] - tenure["first_txn"]).dt.days.clip(lower=0)
tenure["recency_days"] = (overall_last_date - tenure["last_txn"]).dt.days.clip(lower=0)

# Timing ratios
tx["txn_weekday"] = tx["Transaction_Date"].dt.dayofweek
tx["txn_is_weekend"] = tx["txn_weekday"].isin([5, 6]).astype(int)
tx["txn_is_business_hours"] = tx["Transaction_Hour"].between(9, 16).astype(int)

timing = tx.groupby("Customer_ID").agg(
    pct_weekend_txn=("txn_is_weekend", "mean"),
    pct_business_hours_txn=("txn_is_business_hours", "mean"),
    avg_txn_hour=("Transaction_Hour", "mean"),
    std_txn_hour=("Transaction_Hour", "std"),
).reset_index()

timing["std_txn_hour"] = timing["std_txn_hour"].fillna(0)

# Time-of-day distribution (one-hot proportions)
tod = pd.get_dummies(tx[["Customer_ID","Transaction_TimeOfDay"]], columns=["Transaction_TimeOfDay"])
tod = tod.groupby("Customer_ID").mean().reset_index()
tod = tod.rename(columns={c: c.replace("Transaction_TimeOfDay_", "pct_tod_") for c in tod.columns if c.startswith("Transaction_TimeOfDay_")})

display(tenure.head())
display(timing.head())
display(tod.head())


,Customer_ID,first_txn,last_txn,active_days,tenure_days,recency_days
0,1,2023-01-02 04:00:00,2023-07-02 03:00:00,6,180,26
1,2,2023-02-06 04:00:00,2023-05-21 23:00:00,2,104,67
2,3,2023-02-27 23:00:00,2023-02-27 23:00:00,1,0,150
3,4,2023-01-22 15:00:00,2023-06-30 01:00:00,2,158,28
4,5,2023-02-19 15:00:00,2023-07-27 00:00:00,5,157,1


,Customer_ID,pct_weekend_txn,pct_business_hours_txn,avg_txn_hour,std_txn_hour
0,1,0.17,0.17,6.83,6.85
1,2,0.50,0.00,13.50,13.44
2,3,0.00,0.00,23.00,0.00
3,4,0.50,0.50,8.00,9.90
4,5,0.40,0.60,12.80,8.38


,Customer_ID,pct_tod_Afternoon,pct_tod_Evening,pct_tod_Morning,pct_tod_Night
0,1,0.17,0.17,0.00,0.67
1,2,0.00,0.00,0.00,1.00
2,3,0.00,0.00,0.00,1.00
3,4,0.50,0.00,0.00,0.50
4,5,0.40,0.00,0.20,0.40


## 8) Transaction mix and engagement breadth (UNCHANGED)

In [9]:

type_counts = (
    tx.pivot_table(index="Customer_ID", columns="Transaction_Type", values="Transaction_ID", aggfunc="count", fill_value=0)
    .reset_index()
)

type_feature_cols = [c for c in type_counts.columns if c != "Customer_ID"]
type_counts = type_counts.rename(columns={c: f"count_type_{str(c).lower().replace(' ','_')}" for c in type_feature_cols})

type_props = type_counts.copy()
count_cols = [c for c in type_props.columns if c.startswith("count_type_")]
type_props["txn_count_total_for_props"] = type_props[count_cols].sum(axis=1).replace(0, np.nan)

for c in count_cols:
    type_props[c.replace("count_type_", "pct_type_")] = type_props[c] / type_props["txn_count_total_for_props"]

type_props = type_props[["Customer_ID"] + [c for c in type_props.columns if c.startswith("pct_type_")]].fillna(0)

breadth = tx.groupby("Customer_ID")["Transaction_Type"].nunique().reset_index(name="engagement_breadth")

display(type_counts.head())
display(type_props.head())
display(breadth.head())


Transaction_Type,Customer_ID,count_type_bill_payment,count_type_investment,count_type_loan_payment,count_type_purchase
0,1,3,1,1,1
1,2,1,0,1,0
2,3,1,0,0,0
3,4,0,0,1,1
4,5,2,1,0,2


Transaction_Type,Customer_ID,pct_type_bill_payment,pct_type_investment,pct_type_loan_payment,pct_type_purchase
0,1,0.50,0.17,0.17,0.17
1,2,0.50,0.00,0.50,0.00
2,3,1.00,0.00,0.00,0.00
3,4,0.00,0.00,0.50,0.50
4,5,0.40,0.20,0.00,0.40


,Customer_ID,engagement_breadth
0,1,4
1,2,2
2,3,1
3,4,2
4,5,3


## 9) Trend features (monthly spend slope) (UNCHANGED)

In [10]:

monthly_spend = (
    tx.groupby(["Customer_ID", "Transaction_Year", "Transaction_Month"], as_index=False)
      .agg(month_total_spend=("Transaction_Amount", "sum"))
)

monthly_spend = monthly_spend.sort_values(["Customer_ID", "Transaction_Year", "Transaction_Month"])
monthly_spend["month_ord"] = monthly_spend.groupby("Customer_ID").cumcount()

def slope_for_customer(df_cust: pd.DataFrame) -> float:
    if len(df_cust) < 2:
        return 0.0
    x = df_cust["month_ord"].values.astype(float)
    y = df_cust["month_total_spend"].values.astype(float)
    return float(np.polyfit(x, y, 1)[0])

trend = monthly_spend.groupby("Customer_ID").apply(slope_for_customer).reset_index(name="monthly_spend_slope")

last_year = int(tx["Transaction_Year"].max())
last_month = int(tx.loc[tx["Transaction_Year"] == last_year, "Transaction_Month"].max())

last_month_spend = monthly_spend.loc[
    (monthly_spend["Transaction_Year"] == last_year) & (monthly_spend["Transaction_Month"] == last_month),
    ["Customer_ID", "month_total_spend"]
].rename(columns={"month_total_spend": "last_month_spend"})

overall_mean_monthly = monthly_spend.groupby("Customer_ID")["month_total_spend"].mean().reset_index(name="avg_monthly_spend")

trend = trend.merge(last_month_spend, on="Customer_ID", how="left").merge(overall_mean_monthly, on="Customer_ID", how="left")
trend["last_month_vs_avg_monthly"] = np.where(trend["avg_monthly_spend"] > 0, trend["last_month_spend"].fillna(0) / trend["avg_monthly_spend"], 0)

display(trend.head())


,Customer_ID,monthly_spend_slope,last_month_spend,avg_monthly_spend,last_month_vs_avg_monthly
0,1,762.00,"4,993.00","4,209.00",1.19
1,2,793.00,NaN,"2,453.50",0.00
2,3,0.00,NaN,"1,538.00",0.00
3,4,"-1,177.00",NaN,"4,147.50",0.00
4,5,-99.50,"3,708.00","4,932.67",0.75


## 10) Risk proxy and feedback sentiment (domain-driven) (UNCHANGED)

In [11]:

q25 = tx["Transaction_Amount"].quantile(0.25)
q75 = tx["Transaction_Amount"].quantile(0.75)

def risk_proxy(a: float) -> str:
    if a >= q75:
        return "High"
    if a <= q25:
        return "Low"
    return "Medium"

tx["risk_proxy"] = tx["Transaction_Amount"].apply(risk_proxy)

risk = tx.groupby("Customer_ID").agg(
    high_risk_txn_count=("risk_proxy", lambda s: int((s == "High").sum())),
    pct_high_risk_txn=("risk_proxy", lambda s: float((s == "High").mean())),
).reset_index()

sentiment_map = {
    "Excellent": 1.0,
    "Very Satisfied": 1.0,
    "Good Service": 0.5,
    "Needs Improvement": -0.5,
    "Unsatisfactory": -1.0,
}

fb = feedback_agg.copy()
fb["top_comment_sentiment"] = fb["top_comment"].map(sentiment_map).fillna(0)

display(risk.head())
display(fb.head())


,Customer_ID,high_risk_txn_count,pct_high_risk_txn
0,1,3,0.50
1,2,0,0.00
2,3,0,0.00
3,4,1,0.50
4,5,2,0.40


,Customer_ID,feedback_count,satisfaction_mean,satisfaction_std,ltr_mean,ltr_std,top_comment,top_comment_sentiment
0,1,2,8.50,2.12,9.00,0.00,Very Satisfied,1.00
1,2,3,4.33,2.31,5.00,2.00,Very Satisfied,1.00
2,3,3,8.33,2.08,3.67,3.79,Very Satisfied,1.00
3,4,6,5.50,2.07,3.33,2.50,Needs Improvement,-0.50
4,5,9,6.44,2.01,5.67,2.87,Unsatisfactory,-1.00


## 11) Combine all features into a single customer table (UNCHANGED)

In [12]:

customer_features = spend.merge(tenure[["Customer_ID","active_days","tenure_days","recency_days"]], on="Customer_ID", how="left") \
                         .merge(timing, on="Customer_ID", how="left") \
                         .merge(tod, on="Customer_ID", how="left") \
                         .merge(type_counts, on="Customer_ID", how="left") \
                         .merge(type_props, on="Customer_ID", how="left") \
                         .merge(breadth, on="Customer_ID", how="left") \
                         .merge(trend[["Customer_ID","monthly_spend_slope","last_month_vs_avg_monthly"]], on="Customer_ID", how="left") \
                         .merge(risk, on="Customer_ID", how="left") \
                         .merge(fb, on="Customer_ID", how="left")

for col in customer_features.columns:
    if col == "Customer_ID":
        continue
    if customer_features[col].dtype.kind in "biufc":
        customer_features[col] = customer_features[col].fillna(0)
    else:
        customer_features[col] = customer_features[col].fillna("Unknown")

top_comment_dummies = pd.get_dummies(customer_features["top_comment"], prefix="top_comment")
customer_features = pd.concat([customer_features.drop(columns=["top_comment"]), top_comment_dummies], axis=1)

customer_features = customer_features.round(2)

print("Customer feature table shape:", customer_features.shape)
display(customer_features.head())


Customer feature table shape: (993, 46)


,Customer_ID,txn_count,total_spend,avg_spend,median_spend,min_spend,max_spend,std_spend,cv_spend,pct_amount_imputed,pct_amount_outlier,active_days,tenure_days,recency_days,pct_weekend_txn,pct_business_hours_txn,avg_txn_hour,std_txn_hour,pct_tod_Afternoon,pct_tod_Evening,pct_tod_Morning,pct_tod_Night,count_type_bill_payment,count_type_investment,count_type_loan_payment,count_type_purchase,pct_type_bill_payment,pct_type_investment,pct_type_loan_payment,pct_type_purchase,engagement_breadth,monthly_spend_slope,last_month_vs_avg_monthly,high_risk_txn_count,pct_high_risk_txn,feedback_count,satisfaction_mean,satisfaction_std,ltr_mean,ltr_std,top_comment_sentiment,top_comment_Excellent,top_comment_Good Service,top_comment_Needs Improvement,top_comment_Unsatisfactory,top_comment_Very Satisfied
0,1,6,"16,836.00","2,806.00","2,771.00",156.00,"4,993.00","2,062.31",0.73,0.00,0.00,6,180,26,0.17,0.17,6.83,6.85,0.17,0.17,0.00,0.67,3,1,1,1,0.50,0.17,0.17,0.17,4,762.00,1.19,3,0.50,2,8.50,2.12,9.00,0.00,1.00,False,False,False,False,True
1,2,2,"4,907.00","2,453.50","2,453.50","2,057.00","2,850.00",560.74,0.23,0.00,0.00,2,104,67,0.50,0.00,13.50,13.44,0.00,0.00,0.00,1.00,1,0,1,0,0.50,0.00,0.50,0.00,2,793.00,0.00,0,0.00,3,4.33,2.31,5.00,2.00,1.00,False,False,False,False,True
2,3,1,"1,538.00","1,538.00","1,538.00","1,538.00","1,538.00",0.00,0.00,0.00,0.00,1,0,150,0.00,0.00,23.00,0.00,0.00,0.00,0.00,1.00,1,0,0,0,1.00,0.00,0.00,0.00,1,0.00,0.00,0,0.00,3,8.33,2.08,3.67,3.79,1.00,False,False,False,False,True
3,4,2,"8,295.00","4,147.50","4,147.50","3,559.00","4,736.00",832.26,0.20,0.00,0.00,2,158,28,0.50,0.50,8.00,9.90,0.50,0.00,0.00,0.50,0,0,1,1,0.00,0.00,0.50,0.50,2,"-1,177.00",0.00,1,0.50,6,5.50,2.07,3.33,2.50,-0.50,False,False,True,False,False
4,5,5,"14,798.00","2,959.60","2,305.00","1,508.00","4,878.00","1,386.58",0.47,0.00,0.00,5,157,1,0.40,0.60,12.80,8.38,0.40,0.00,0.20,0.40,2,1,0,2,0.40,0.20,0.00,0.40,3,-99.50,0.75,2,0.40,9,6.44,2.01,5.67,2.87,-1.00,False,False,False,True,False


## 12) Scaling: create a clustering-ready numeric matrix (UNCHANGED)

In [13]:

from sklearn.preprocessing import StandardScaler

numeric_cols = [c for c in customer_features.columns if c != "Customer_ID" and customer_features[c].dtype.kind in "biufc"]

scaler = StandardScaler()
scaled_values = scaler.fit_transform(customer_features[numeric_cols])

customer_features_scaled = pd.DataFrame(scaled_values, columns=numeric_cols)
customer_features_scaled.insert(0, "Customer_ID", customer_features["Customer_ID"].values)

display(customer_features_scaled.head())


,Customer_ID,txn_count,total_spend,avg_spend,median_spend,min_spend,max_spend,std_spend,cv_spend,pct_amount_imputed,pct_amount_outlier,active_days,tenure_days,recency_days,pct_weekend_txn,pct_business_hours_txn,avg_txn_hour,std_txn_hour,pct_tod_Afternoon,pct_tod_Evening,pct_tod_Morning,pct_tod_Night,count_type_bill_payment,count_type_investment,count_type_loan_payment,count_type_purchase,pct_type_bill_payment,pct_type_investment,pct_type_loan_payment,pct_type_purchase,engagement_breadth,monthly_spend_slope,last_month_vs_avg_monthly,high_risk_txn_count,pct_high_risk_txn,feedback_count,satisfaction_mean,satisfaction_std,ltr_mean,ltr_std,top_comment_sentiment,top_comment_Excellent,top_comment_Good Service,top_comment_Needs Improvement,top_comment_Unsatisfactory,top_comment_Very Satisfied
0,1,0.44,0.66,0.41,0.32,-0.91,0.97,1.48,0.62,-0.30,-0.09,0.47,1.00,-0.35,-0.49,-0.69,-1.29,0.24,-0.18,0.04,-1.24,1.33,1.51,-0.23,-0.25,-0.23,1.13,-0.38,-0.36,-0.38,1.26,0.41,1.19,1.54,1.08,-1.50,2.06,-0.49,2.36,-2.67,1.15,-0.28,-0.74,-0.51,-0.48,2.15
1,2,-1.39,-1.19,-0.06,-0.01,1.12,-1.23,-1.41,-1.24,-0.30,-0.09,-1.38,-0.48,0.78,0.98,-1.40,0.54,2.88,-1.02,-0.89,-1.24,2.65,-0.22,-1.17,-0.25,-1.17,1.13,-1.15,1.15,-1.20,-1.00,0.43,-0.75,-1.12,-1.11,-0.99,-0.91,-0.30,-0.39,-0.63,1.15,-0.28,-0.74,-0.51,-0.48,2.15
2,3,-1.85,-1.72,-1.27,-0.98,0.57,-2.58,-2.49,-2.10,-0.30,-0.09,-1.85,-2.50,3.06,-1.25,-1.40,3.16,-2.50,-1.02,-0.89,-1.24,2.65,-0.22,-1.17,-1.11,-1.17,3.35,-1.15,-1.13,-1.20,-2.13,-0.03,-0.75,-1.12,-1.11,-0.99,1.94,-0.53,-1.31,1.19,1.15,-0.28,-0.74,-0.51,-0.48,2.15
3,4,-1.39,-0.67,2.18,1.77,2.73,0.70,-0.89,-1.35,-0.30,-0.09,-1.38,0.57,-0.29,0.98,0.70,-0.97,1.46,1.45,-0.89,-1.24,0.65,-1.08,-1.17,-0.25,-0.23,-1.09,-1.15,1.15,1.20,-1.00,-0.72,-0.75,-0.23,1.08,0.52,-0.08,-0.54,-1.54,-0.12,-0.86,-0.28,-0.74,1.94,-0.48,-0.46
4,5,-0.02,0.35,0.61,-0.17,0.54,0.85,0.18,-0.35,-0.30,-0.09,0.01,0.56,-1.04,0.53,1.12,0.35,0.85,0.95,-0.89,-0.39,0.25,0.65,-0.23,-1.11,0.71,0.68,-0.25,-1.13,0.72,0.13,-0.09,0.47,0.65,0.65,2.03,0.59,-0.60,0.07,0.25,-1.53,-0.28,-0.74,-0.51,2.10,-0.46


## 13) Feature dictionary (brief) (UNCHANGED)

In [14]:

feature_dict = []

def add_feature(name, description, group):
    feature_dict.append({"feature": name, "group": group, "description": description})

# Spending
add_feature("txn_count", "Total number of transactions for the customer.", "Spending")
add_feature("total_spend", "Sum of transaction amounts.", "Spending")
add_feature("avg_spend", "Mean transaction amount.", "Spending")
add_feature("median_spend", "Median transaction amount.", "Spending")
add_feature("min_spend", "Minimum transaction amount.", "Spending")
add_feature("max_spend", "Maximum transaction amount.", "Spending")
add_feature("std_spend", "Standard deviation of transaction amounts.", "Spending")
add_feature("cv_spend", "Coefficient of variation (std/mean) as spend stability indicator.", "Spending")
add_feature("pct_amount_imputed", "Share of transactions where amount was imputed during cleaning.", "DataQuality")
add_feature("pct_amount_outlier", "Share of transactions flagged as outliers before IQR-capping.", "DataQuality")

# Temporal
add_feature("active_days", "Number of unique transaction days.", "Temporal")
add_feature("tenure_days", "Days between first and last transaction.", "Temporal")
add_feature("recency_days", "Days since last transaction (relative to global last date).", "Temporal")
add_feature("pct_weekend_txn", "Share of transactions occurring on weekends.", "Temporal")
add_feature("pct_business_hours_txn", "Share of transactions during 9AM–5PM.", "Temporal")
add_feature("avg_txn_hour", "Average transaction hour (0–23).", "Temporal")
add_feature("std_txn_hour", "Std dev of transaction hour (timing variability).", "Temporal")
add_feature("pct_tod_Morning", "Share of transactions in Morning bucket.", "Temporal")
add_feature("pct_tod_Afternoon", "Share of transactions in Afternoon bucket.", "Temporal")
add_feature("pct_tod_Evening", "Share of transactions in Evening bucket.", "Temporal")
add_feature("pct_tod_Night", "Share of transactions in Night bucket.", "Temporal")

# Mix/Engagement
add_feature("engagement_breadth", "Number of distinct transaction types used.", "Engagement")
add_feature("monthly_spend_slope", "Slope of customer monthly spend over time (positive=increasing).", "Trend")
add_feature("last_month_vs_avg_monthly", "Ratio of last-month spend vs customer’s average monthly spend.", "Trend")
add_feature("high_risk_txn_count", "Count of high-amount (≥75th percentile) transactions.", "RiskProxy")
add_feature("pct_high_risk_txn", "Share of high-amount (≥75th percentile) transactions.", "RiskProxy")
add_feature("top_comment_sentiment", "Sentiment score derived from the customer’s most common feedback category.", "Feedback")

# Feedback stats
add_feature("feedback_count", "Number of feedback records associated with the customer.", "Feedback")
add_feature("satisfaction_mean", "Mean satisfaction score (customer-level).", "Feedback")
add_feature("satisfaction_std", "Variation of satisfaction score.", "Feedback")
add_feature("ltr_mean", "Mean likelihood-to-recommend score (customer-level).", "Feedback")
add_feature("ltr_std", "Variation of likelihood-to-recommend score.", "Feedback")

feature_dictionary = pd.DataFrame(feature_dict)
display(feature_dictionary.head(20))


,feature,group,description
0,txn_count,Spending,Total number of transactions for the customer.
1,total_spend,Spending,Sum of transaction amounts.
2,avg_spend,Spending,Mean transaction amount.
3,median_spend,Spending,Median transaction amount.
4,min_spend,Spending,Minimum transaction amount.
5,max_spend,Spending,Maximum transaction amount.
6,std_spend,Spending,Standard deviation of transaction amounts.
7,cv_spend,Spending,Coefficient of variation (std/mean) as spend s...
8,pct_amount_imputed,DataQuality,Share of transactions where amount was imputed...
9,pct_amount_outlier,DataQuality,Share of transactions flagged as outliers befo...


## 14) Export to `features/` folder

Exports are intentionally simple and stable (no timestamps).


In [15]:

out_dir = Path("features")
out_dir.mkdir(exist_ok=True)

customer_features.to_csv(out_dir / "customer_features.csv", index=False)
customer_features_scaled.to_csv(out_dir / "customer_features_scaled.csv", index=False)
feature_dictionary.to_csv(out_dir / "feature_dictionary.csv", index=False)
tx_time_features.to_csv(out_dir / "transactions_time_features.csv", index=False)
merged_master_transactions.to_csv(out_dir / "merged_master_transactions.csv", index=False)

print("Saved:")
print("-", out_dir / "customer_features.csv")
print("-", out_dir / "customer_features_scaled.csv")
print("-", out_dir / "feature_dictionary.csv")
print("-", out_dir / "transactions_time_features.csv")
print("-", out_dir / "merged_master_transactions.csv")

Saved:
- features/customer_features.csv
- features/customer_features_scaled.csv
- features/feature_dictionary.csv
- features/transactions_time_features.csv
- features/merged_master_transactions.csv


## 15) Quick validation checks

In [16]:

missing = customer_features.isna().sum()
print("Missing values (non-zero):")
display(missing[missing > 0])

display(customer_features.describe().round(2))


Missing values (non-zero):


Series([], dtype: int64)

,Customer_ID,txn_count,total_spend,avg_spend,median_spend,min_spend,max_spend,std_spend,cv_spend,pct_amount_imputed,pct_amount_outlier,active_days,tenure_days,recency_days,pct_weekend_txn,pct_business_hours_txn,avg_txn_hour,std_txn_hour,pct_tod_Afternoon,pct_tod_Evening,pct_tod_Morning,pct_tod_Night,count_type_bill_payment,count_type_investment,count_type_loan_payment,count_type_purchase,pct_type_bill_payment,pct_type_investment,pct_type_loan_payment,pct_type_purchase,engagement_breadth,monthly_spend_slope,last_month_vs_avg_monthly,high_risk_txn_count,pct_high_risk_txn,feedback_count,satisfaction_mean,satisfaction_std,ltr_mean,ltr_std,top_comment_sentiment
count,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00,993.00
mean,500.87,5.04,"12,569.72","2,497.78","2,467.55","1,007.09","4,048.93","1,295.00",0.56,0.02,0.00,4.98,128.50,38.72,0.28,0.33,11.52,6.25,0.21,0.16,0.29,0.34,1.25,1.25,1.29,1.25,0.25,0.25,0.25,0.25,2.89,60.25,0.46,1.26,0.25,4.97,5.61,2.60,5.57,2.62,0.14
std,288.63,2.19,"6,424.66",758.16,951.59,933.77,975.53,520.14,0.27,0.06,0.02,2.15,51.35,36.40,0.23,0.24,3.63,2.50,0.20,0.18,0.24,0.25,1.16,1.07,1.16,1.06,0.23,0.22,0.22,0.21,0.88,"1,723.03",0.61,1.13,0.23,1.98,1.41,0.98,1.46,0.98,0.75
min,1.00,1.00,286.00,286.00,167.00,10.00,286.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,"-11,148.00",0.00,0.00,0.00,1.00,1.00,0.00,1.00,0.00,-1.00
25%,251.00,3.00,"7,777.00","2,015.00","1,821.00",324.00,"3,592.00","1,004.00",0.40,0.00,0.00,3.00,98.00,11.00,0.11,0.17,9.25,4.89,0.00,0.00,0.11,0.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,-672.00,0.00,0.00,0.00,4.00,4.75,2.07,4.57,2.08,-0.50
50%,502.00,5.00,"11,939.00","2,458.67","2,440.00",745.00,"4,347.00","1,319.17",0.56,0.00,0.00,5.00,139.00,28.00,0.25,0.33,11.33,6.56,0.20,0.14,0.25,0.33,1.00,1.00,1.00,1.00,0.22,0.25,0.25,0.25,3.00,0.00,0.00,1.00,0.25,5.00,5.60,2.70,5.67,2.72,0.50
75%,750.00,6.00,"16,531.00","2,930.75","3,124.50","1,369.00","4,734.00","1,634.15",0.72,0.00,0.00,6.00,169.00,56.00,0.43,0.50,13.62,7.94,0.33,0.25,0.44,0.50,2.00,2.00,2.00,2.00,0.38,0.38,0.40,0.40,4.00,748.50,0.86,2.00,0.38,6.00,6.50,3.25,6.50,3.30,1.00
max,"1,000.00",13.00,"37,281.00","4,993.00","4,993.00","4,993.00","7,315.00","3,335.42",1.73,0.50,0.33,13.00,205.00,197.00,1.00,1.00,23.00,14.14,1.00,1.00,1.00,1.00,7.00,5.00,7.00,6.00,1.00,1.00,1.00,1.00,4.00,"11,068.00",3.11,6.00,1.00,13.00,10.00,6.36,10.00,6.36,1.00


## Summary and Insights (Feature Engineering)

- **Goal:** Convert cleaned FinMark datasets into **model-ready features** for segmentation/clustering and analysis.

- **Time normalization applied (Transactions):**
  - Standardized `Transaction_Date` to datetime.
  - Extracted **Day / Month / Year / ISO Week** and **Transaction_Hour**.
  - Created **Transaction_TimeOfDay** buckets (Morning/Afternoon/Evening/Night).

- **Trend features engineered (grouped correctly):**
  - `Daily_Spend_Trend` = total spend per **Customer_ID + Transaction_Day**
  - `Weekly_Spend_Trend` = total spend per **Customer_ID + ISOYear + ISOWeek**
  - Exported as `features/transactions_time_features.csv` for documentation and additional EDA.

- **Model-friendly master dataset created (no duplication):**
  - Built `features/merged_master_transactions.csv` by merging:
    - transaction-level time features + **customer-level feedback summary**
  - Keeps row count aligned with transactions and avoids feedback-driven row inflation.

- **Customer-level segmentation dataset produced:**
  - `features/customer_features.csv` (1 row per customer) including:
    - spending intensity/variability, recency/tenure, timing patterns, transaction-type mix, and risk proxy
    - aggregated feedback metrics + sentiment score
  - `features/customer_features_scaled.csv` created for distance-based clustering (standardized numeric features).

- **Key takeaway:** Outputs are separated into **transaction-level (documentation/analysis)** and **customer-level (modeling)** datasets to ensure clarity, reproducibility, and readiness for the next modeling phase.
